# Step 2: Activation Extraction

## Corporate Identity Awareness in LLMs

In this notebook we extract **hidden-state activations** from Gemma-2-9B-IT across all
six identity conditions. These activations form the foundation for everything that
follows: linear probing, steering, and statistical analysis.

### What this notebook covers

1. **Loading the model** -- Gemma-2-9B-IT via HuggingFace with `output_hidden_states=True`.
2. **Prompt formatting** -- how identity system prompts are injected into Gemma's chat template.
3. **Single activation extraction** -- understanding the tensor shapes and what we capture.
4. **Batch extraction** -- running all identity x query combinations through the model.
5. **Normalization** -- zero-mean, unit-variance per layer for downstream analysis.
6. **Saving** -- persisting activations to disk for reuse.
7. **Sanity checks** -- verifying the activations look reasonable.

### Key idea

For each (identity, query) pair we run a **forward pass** through the model and capture
the residual-stream activation at the **last token position** across all 42 transformer
layers. This gives us a tensor of shape `(42, 3584)` per sample -- a rich representation
of how the model processes the query under a given identity condition.

### Prerequisites

- A GPU with sufficient VRAM (A40 / A100 recommended for Gemma-2-9B in bfloat16).
- The dataset constructed in Notebook 01.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt

from research.config import (
    model_config, experiment_config,
    IDENTITY_CONDITIONS,
    ACTIVATIONS_DIR,
)
from research.models.loader import ModelLoader
from research.models.activation_extractor import ActivationExtractor
from research.data.prompts import ALL_QUERIES, QUERY_CATEGORIES
from research.data.dataset import ContrastiveDataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2.1 Loading the Model

We use **Gemma-2-9B-IT** (`google/gemma-2-9b-it`) as our primary model. This is a
particularly interesting choice for studying corporate identity awareness because:

- It is made by **Google DeepMind**, so the `google` identity condition is the model's
  *actual* creator -- while `anthropic`, `openai`, and `meta` are foreign identities.
  This lets us test whether models respond differently to their true vs. false corporate
  identity.
- At 9B parameters with 42 transformer layers and a 3584 hidden dimension, it is large
  enough to exhibit sophisticated identity-conditioned behavior yet small enough to run
  activation extraction on a single GPU.
- The instruction-tuned variant (`-it`) has been trained to follow system prompts,
  making it responsive to identity conditioning.

The `ModelLoader` handles loading with `output_hidden_states=True` so that every
forward pass returns the residual stream at each layer. If Gemma-2-9B fails to load
(e.g., insufficient VRAM), it automatically falls back to Qwen2.5-7B-Instruct.

In [ ]:
# Initialize the loader and load the model + tokenizer
loader = ModelLoader(config=model_config)
model, tokenizer = loader.load_model()

# Print model information
info = loader.get_model_info()
print("=== Model Info ===")
for key, value in info.items():
    print(f"  {key:15s}: {value}")
print()

# Quick architecture overview
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e9:.2f}B")
print(f"Model class     : {type(model).__name__}")

# Show the top-level modules
print("\n=== Top-Level Architecture ===")
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}: {n_params / 1e6:>8.1f}M params")

## 2.2 Understanding Prompt Formatting

Gemma-2-IT uses a specific chat template with `<start_of_turn>` / `<end_of_turn>` tags.
Crucially, Gemma's chat format has **no dedicated system role** -- unlike models from
Anthropic or OpenAI. Our `ModelLoader.format_prompt()` method handles this by prepending
the system prompt to the user turn:

```
<start_of_turn>user
{system_prompt}

{user_query}<end_of_turn>
<start_of_turn>model
```

This means the identity information flows through the *same attention path* as the
user query, rather than being segregated into a separate system channel. This is
important context for interpreting our activation results -- the model must learn to
separate identity from query content within its residual stream.

Let's see what the formatted prompts look like for each identity condition.

In [ ]:
sample_query = "Tell me about yourself"

print("=== Formatted Prompts for Each Identity ===")
for identity_label, system_prompt in IDENTITY_CONDITIONS.items():
    formatted = loader.format_prompt(system_prompt, sample_query)
    
    # Count tokens for reference
    tokens = tokenizer(formatted, return_tensors="pt")
    n_tokens = tokens["input_ids"].shape[1]
    
    print(f"\n--- {identity_label} ({n_tokens} tokens) ---")
    print(formatted)
    print("---")

## 2.3 Single Activation Extraction

Before running batch extraction, let's understand exactly what we are capturing.

The `ActivationExtractor.extract_activations()` method:

1. Formats the prompt using the model's chat template.
2. Tokenizes and runs a **forward pass** with `output_hidden_states=True`.
3. Collects the hidden states from all 42 transformer layers (excluding the embedding
   layer at index 0).
4. Extracts the activation vector at the **last token position** (by default).

The result is a tensor of shape `(num_layers, hidden_dim)` = `(42, 3584)` for Gemma-2-9B.

**Why the last token?** In autoregressive transformers, the last token position
aggregates information from the entire context via causal attention. It is the position
where the model has "seen" both the identity prompt and the query, making it the richest
single-position summary of how the model represents the full input.

Alternative strategies (`"mean"` for mean-pooling all tokens, or `"system_prompt_mean"`
for pooling only system-prompt tokens) are also supported.

In [ ]:
# Create the extractor
extractor = ActivationExtractor(model, tokenizer, config=experiment_config)

# Extract activations for a single example
sample_identity = "anthropic"
sample_system = IDENTITY_CONDITIONS[sample_identity]
sample_query = "Tell me about yourself"

print(f"Identity : {sample_identity}")
print(f"Query    : {sample_query}")
print(f"Extracting activations...")

acts = extractor.extract_activations(
    system_prompt=sample_system,
    user_query=sample_query,
    token_position="last",
)

print(f"\nActivation tensor shape: {acts.shape}")
print(f"  = ({model_config.num_layers} layers, {model_config.hidden_dim} hidden dims)")
print(f"Dtype: {acts.dtype}")
print()

# Show activation norms per layer -- these give a sense of "how active"
# each layer is for this input
print("=== Activation L2 Norms per Layer ===")
norms = torch.norm(acts, dim=1)
for layer_idx in range(0, len(norms), 6):  # every 6th layer for readability
    print(f"  Layer {layer_idx:2d}: {norms[layer_idx]:.2f}")
print(f"  Layer {len(norms)-1:2d}: {norms[-1]:.2f}  (final layer)")
print(f"\nMin norm : {norms.min():.2f} (layer {norms.argmin().item()})")
print(f"Max norm : {norms.max():.2f} (layer {norms.argmax().item()})")

## 2.4 Batch Extraction Across All Conditions

Now we extract activations for the **full experimental matrix**: every identity condition
crossed with every query. This is the most computationally expensive step in the pipeline.

The `extract_all_conditions()` method:
- Takes a list of queries and a dict of identity conditions.
- Builds all (identity, query) samples internally.
- Runs forward passes with a progress bar.
- Returns a nested dict: `activations[identity][query] -> tensor(42, 3584)`.

**Expected runtime**: With ~60 queries x 6 identities = 360 forward passes, this
typically takes 5-15 minutes on an A40/A100 depending on sequence lengths.

In [ ]:
# Build the dataset
dataset = ContrastiveDataset()
print(repr(dataset))
print(f"Total forward passes needed: {len(dataset)}")
print()

# Extract activations for all conditions
print("Starting extraction (this may take several minutes)...")
print()

activations = extractor.extract_all_conditions(
    queries=dataset.queries,
    identities=dataset.identities,
    token_position=experiment_config.token_position,  # "last" by default
)

# Print summary
print("\n=== Extraction Complete ===")
for identity, query_acts in activations.items():
    sample_tensor = next(iter(query_acts.values()))
    print(f"  {identity:10s}: {len(query_acts)} queries, tensor shape = {sample_tensor.shape}")

total_samples = sum(len(v) for v in activations.values())
print(f"\nTotal activation tensors: {total_samples}")
print(f"Memory per tensor       : {sample_tensor.nelement() * sample_tensor.element_size() / 1024:.1f} KB")
total_mb = total_samples * sample_tensor.nelement() * sample_tensor.element_size() / (1024 * 1024)
print(f"Total memory (approx)   : {total_mb:.1f} MB")

## 2.5 Normalization

Before using these activations for probing or steering, we apply **per-layer
normalization**: zero mean and unit variance.

**Why normalize?**

- Different layers operate at different scales. Early layers may have smaller activation
  norms than later layers. Without normalization, a linear probe trained across layers
  would be dominated by whichever layer has the largest raw magnitudes.
- Normalization ensures that each layer contributes proportionally based on its
  *directional* information (which directions carry identity signal) rather than its
  raw scale.
- It also improves numerical stability for downstream logistic regression and PCA.

The `normalize_activations()` method computes the mean and standard deviation across
all samples and hidden dimensions *per layer*, then applies `(x - mean) / std`.

In [ ]:
# Normalize activations
normalized_activations = ActivationExtractor.normalize_activations(activations)

# Verify normalization: check that each layer has approximately zero mean
# and unit variance across samples
print("=== Normalization Verification ===")
print("Collecting per-layer statistics across all normalized samples...\n")

all_tensors = []
for identity_acts in normalized_activations.values():
    for tensor in identity_acts.values():
        all_tensors.append(tensor)

stacked = torch.stack(all_tensors, dim=0)  # (n_samples, num_layers, hidden_dim)
print(f"Stacked shape: {stacked.shape}")
print()

# Per-layer stats
for layer_idx in [0, 10, 20, 30, 41]:
    layer_data = stacked[:, layer_idx, :]  # (n_samples, hidden_dim)
    mean = layer_data.mean().item()
    std = layer_data.std().item()
    print(f"  Layer {layer_idx:2d}: mean = {mean:+.4f}, std = {std:.4f}")

print("\n(Values should be close to mean=0, std=1)")

## 2.6 Save Activations

We save both raw and normalized activations to disk so that subsequent notebooks
(probing, steering, analysis) can load them without re-running the expensive
forward passes.

Files are saved as PyTorch `.pt` files in `outputs/activations/`. The nested dict
structure `activations[identity][query] -> tensor` is preserved exactly.

In [ ]:
from pathlib import Path

# Define output paths
raw_path = ACTIVATIONS_DIR / "raw_activations.pt"
norm_path = ACTIVATIONS_DIR / "normalized_activations.pt"

# Save raw activations
ActivationExtractor.save_activations(activations, raw_path)
print(f"Raw activations saved to: {raw_path}")

# Save normalized activations
ActivationExtractor.save_activations(normalized_activations, norm_path)
print(f"Normalized activations saved to: {norm_path}")

# Print file sizes
print("\n=== File Sizes ===")
for fpath in [raw_path, norm_path]:
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  {fpath.name:40s}: {size_mb:.1f} MB")

# Verify we can reload them
print("\n=== Reload Verification ===")
reloaded = ActivationExtractor.load_activations(norm_path)
print(f"Reloaded identities: {list(reloaded.keys())}")
for identity, query_acts in reloaded.items():
    print(f"  {identity}: {len(query_acts)} queries")
print("Reload successful!")

## 2.7 Quick Sanity Check

As a final sanity check, let's visualize how activation norms vary across layers
for different identity conditions. If the identity system prompt has any effect at all,
we might expect to see divergence in certain layers -- particularly in the middle-to-late
layers where higher-level semantic features are typically represented.

This is not a rigorous analysis (that comes in the probing notebooks), but it provides
a useful "sniff test" that our extraction pipeline is working correctly and that
identity information is indeed leaving a trace in the activations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: Mean activation norm per layer per identity (raw) ────────
ax = axes[0]
for identity, query_acts in activations.items():
    tensors = list(query_acts.values())
    stacked = torch.stack(tensors, dim=0)  # (n_queries, num_layers, hidden_dim)
    norms = torch.norm(stacked, dim=2).mean(dim=0)  # (num_layers,)
    ax.plot(range(len(norms)), norms.numpy(), label=identity, alpha=0.8)

ax.set_xlabel("Layer")
ax.set_ylabel("Mean L2 Norm")
ax.set_title("Raw Activation Norms by Layer and Identity")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 2: Mean activation norm per layer per identity (normalized) ─
ax = axes[1]
for identity, query_acts in normalized_activations.items():
    tensors = list(query_acts.values())
    stacked = torch.stack(tensors, dim=0)
    norms = torch.norm(stacked, dim=2).mean(dim=0)
    ax.plot(range(len(norms)), norms.numpy(), label=identity, alpha=0.8)

ax.set_xlabel("Layer")
ax.set_ylabel("Mean L2 Norm")
ax.set_title("Normalized Activation Norms by Layer and Identity")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ACTIVATIONS_DIR / "activation_norms_by_identity.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nFigure saved to:", ACTIVATIONS_DIR / "activation_norms_by_identity.png")
print("\n" + "=" * 60)
print("Activation extraction complete!")
print("Proceed to Notebook 03 for linear probing analysis.")
print("=" * 60)